In [1]:
import pandas as pd
import os

In [8]:
# Define paths to your downloaded files
NASA_POWER_PATH = "../datasets/nasa_power/nasa_power_climate_risk_indices_190_capitals_1990_2024.csv"
POWER_PLANTS_PATH = "../datasets/openstreetmap/globalpowerplantdatabasev120/global_power_plant_database.csv"

print("--- Checking NASA POWER Dataset ---")
if os.path.exists(NASA_POWER_PATH):
    df_nasa = pd.read_csv(NASA_POWER_PATH, nrows=5)
    print("Success! NASA POWER file read correctly.")
    print("Columns available:", df_nasa.columns.tolist())
else:
    print(f"Error: File not found. Looked at: {os.path.abspath(NASA_POWER_PATH)}")

print("\n--- Checking Infrastructure Dataset ---")
if os.path.exists(POWER_PLANTS_PATH):
    df_plants = pd.read_csv(POWER_PLANTS_PATH, nrows=5)
    print("Success! Infrastructure file read correctly.")
    print("Columns available:", df_plants.columns.tolist())
else:
    print(f"Error: File not found. Looked at: {os.path.abspath(POWER_PLANTS_PATH)}")

--- Checking NASA POWER Dataset ---
Success! NASA POWER file read correctly.
Columns available: ['city', 'iso_alpha3', 'latitude', 'longitude', 'continent', 'who_region', 'wb_income_group', 'year', 'data_days', 'temp_mean_c', 'temp_max_c', 'temp_min_c', 'temp_std_c', 'temp_range_mean_c', 'temp_seasonal_amp', 'temp_yoy_change', 'temp_5yr_mean', 'temp_anomaly', 'days_above_35c', 'days_above_40c', 'days_below_0c', 'days_below_minus10c', 'heat_stress_index', 'cold_stress_index', 'drought_stress_days', 'ideal_climate_days', 'precip_total_mm', 'precip_mean_mm', 'precip_max_day_mm', 'precip_std_mm', 'precip_yoy_change', 'days_heavy_rain', 'days_moderate_rain', 'days_dry', 'rh_mean_pct', 'rh_max_pct', 'dewpoint_mean_c', 'humidity_stress_days', 'wind_mean_ms', 'wind_max_ms', 'wind_std_ms', 'high_wind_days', 'wind_power_density', 'solar_total_mj', 'solar_mean_mj', 'solar_clear_mean_mj', 'solar_clearness_idx', 'solar_peak_days', 'solar_annual_kwh_m2', 'pressure_mean_kpa', 'pressure_std_kpa', 'cli

In [9]:
# Create a targeted dataframe for your Weather/Energy metrics
solar_wind_core = df_nasa[[
    'city', 'latitude', 'longitude', 'year', 
    'wind_mean_ms', 'wind_power_density', 
    'solar_mean_mj', 'solar_annual_kwh_m2'
]]

# Create a targeted dataframe for your Grid Infrastructure metrics
grid_core = df_plants[[
    'name', 'country_long', 'latitude', 'longitude', 
    'primary_fuel', 'capacity_mw'
]]

print("--- Cleaned Solar/Wind Dataset Preview ---")
print(solar_wind_core.head(3))

print("\n--- Cleaned Grid Infrastructure Preview ---")
print(grid_core.head(3))

print("\n--- Quick Statistics for Energy Metrics ---")
print(solar_wind_core[['wind_mean_ms', 'solar_annual_kwh_m2']].describe())

--- Cleaned Solar/Wind Dataset Preview ---
        city  latitude  longitude  year  wind_mean_ms  wind_power_density  \
0  Abu Dhabi     24.47      54.37  1990      3.895288             36.2013   
1  Abu Dhabi     24.47      54.37  1991      3.864548             35.3510   
2  Abu Dhabi     24.47      54.37  1992      3.956475             37.9343   

   solar_mean_mj  solar_annual_kwh_m2  
0       5.713163               579.30  
1       5.487168               556.38  
2       5.377236               546.73  

--- Cleaned Grid Infrastructure Preview ---
                                               name country_long  latitude  \
0      Kajaki Hydroelectric Power Plant Afghanistan  Afghanistan    32.322   
1     Mahipar Hydroelectric Power Plant Afghanistan  Afghanistan    34.556   
2  Naghlu Dam Hydroelectric Power Plant Afghanistan  Afghanistan    34.641   

   longitude primary_fuel  capacity_mw  
0    65.1190        Hydro         33.0  
1    69.4787        Hydro         66.0  
2    69

In [10]:
# Group by city and calculate the average for weather/energy metrics
# We also include latitude and longitude by taking their first occurrence (since coordinates don't change)
city_energy_profiles = df_nasa.groupby('city').agg({
    'latitude': 'first',
    'longitude': 'first',
    'wind_mean_ms': 'mean',
    'wind_power_density': 'mean',
    'solar_mean_mj': 'mean',
    'solar_annual_kwh_m2': 'mean'
}).reset_index()

print("--- Aggregated City Energy Profiles ---")
print(city_energy_profiles.head())
print(f"\nTotal unique cities analyzed: {len(city_energy_profiles)}")

--- Aggregated City Energy Profiles ---
        city  latitude  longitude  wind_mean_ms  wind_power_density  \
0  Abu Dhabi     24.47      54.37      3.930512            37.21344   

   solar_mean_mj  solar_annual_kwh_m2  
0       5.557026              563.764  

Total unique cities analyzed: 1


In [11]:
# 1. Load the FULL datasets (removing nrows=5)
df_nasa_full = pd.read_csv(NASA_POWER_PATH)
df_plants_full = pd.read_csv(POWER_PLANTS_PATH)

# 2. Re-run the aggregation on the full dataset
city_energy_profiles = df_nasa_full.groupby('city').agg({
    'latitude': 'first',
    'longitude': 'first',
    'wind_mean_ms': 'mean',
    'wind_power_density': 'mean',
    'solar_mean_mj': 'mean',
    'solar_annual_kwh_m2': 'mean'
}).reset_index()

print("--- Full Dataset Aggregation Complete ---")
print(f"Total unique cities analyzed now: {len(city_energy_profiles)}")
print("\nTop 5 cities preview:")
print(city_energy_profiles.head())

--- Full Dataset Aggregation Complete ---
Total unique cities analyzed now: 190

Top 5 cities preview:
          city  latitude  longitude  wind_mean_ms  wind_power_density  \
0    Abu Dhabi     24.47      54.37      3.879992           35.844160   
1        Abuja      9.07       7.40      2.374511            8.242749   
2        Accra      5.56      -0.20      3.461180           25.484714   
3  Addis Ababa      9.02      38.74      2.974531           16.205000   
4      Algiers     36.74       3.06      3.509845           26.619926   

   solar_mean_mj  solar_annual_kwh_m2  
0       5.785609           587.054857  
1       5.328320           540.609714  
2       4.930779           500.318000  
3       5.933025           602.014857  
4       4.834681           490.566857  


In [12]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371 # Radius of Earth in kilometers
    return c * r

# Find the distance to the absolute closest power plant for each city
closest_distances = []

for idx, city in city_energy_profiles.iterrows():
    # Calculate distance from this city to EVERY power plant in the world database
    distances = haversine_distance(
        city['latitude'], city['longitude'],
        df_plants_full['latitude'], df_plants_full['longitude']
    )
    # Grab the minimum distance found
    closest_distances.append(distances.min())

# Add the new proximity feature back to our main DataFrame
city_energy_profiles['distance_to_grid_km'] = closest_distances

print("\n--- Geospatial Grid Proximity Added ---")
print(city_energy_profiles[['city', 'latitude', 'longitude', 'distance_to_grid_km']].head())


--- Geospatial Grid Proximity Added ---
          city  latitude  longitude  distance_to_grid_km
0    Abu Dhabi     24.47      54.37            12.769682
1        Abuja      9.07       7.40           117.912753
2        Accra      5.56      -0.20            23.374701
3  Addis Ababa      9.02      38.74            73.530815
4      Algiers     36.74       3.06             2.219781


In [13]:
# Sort by distance_to_grid_km in ascending order (closest first)
best_infrastructure_sites = city_energy_profiles.sort_values(by='distance_to_grid_km', ascending=True)

print("--- Top 5 Cities Closest to the Power Grid ---")
print(best_infrastructure_sites[['city', 'distance_to_grid_km']].head())

print("\n--- Top 5 Cities Furthest from the Power Grid (High Connection Costs) ---")
print(best_infrastructure_sites[['city', 'distance_to_grid_km']].tail())

--- Top 5 Cities Closest to the Power Grid ---
            city  distance_to_grid_km
35      Brussels             0.095264
66      Helsinki             0.260438
27          Bern             0.289112
47    Copenhagen             0.446566
124  Ouagadougou             0.511572

--- Top 5 Cities Furthest from the Power Grid (High Connection Costs) ---
        city  distance_to_grid_km
67   Honiara          1029.623371
125  Palikir          1273.559723
187    Yaren          1611.823360
164   Tarawa          2096.355525
94    Majuro          2369.590760


In [14]:
# 1. Normalize Solar (Higher is better: 1 = Max Solar, 0 = Min Solar)
s_min, s_max = city_energy_profiles['solar_annual_kwh_m2'].min(), city_energy_profiles['solar_annual_kwh_m2'].max()
city_energy_profiles['solar_norm'] = (city_energy_profiles['solar_annual_kwh_m2'] - s_min) / (s_max - s_min)

# 2. Normalize Wind (Higher is better: 1 = Max Wind, 0 = Min Wind)
w_min, w_max = city_energy_profiles['wind_mean_ms'].min(), city_energy_profiles['wind_mean_ms'].max()
city_energy_profiles['wind_norm'] = (city_energy_profiles['wind_mean_ms'] - w_min) / (w_max - w_min)

# 3. Normalize Distance (LOWER is better: 1 = Right next to grid, 0 = Way too far)
d_min, d_max = city_energy_profiles['distance_to_grid_km'].min(), city_energy_profiles['distance_to_grid_km'].max()
city_energy_profiles['grid_norm'] = 1 - ((city_energy_profiles['distance_to_grid_km'] - d_min) / (d_max - d_min))

# 4. Define weights for your AI Engine (Must add up to 1.0)
W_SOLAR = 0.40  # 40% weight on solar potential
W_WIND = 0.30   # 30% weight on wind potential
W_GRID = 0.30   # 30% weight on proximity to infrastructure

# 5. Calculate the Final Composite Suitability Score (Scaled 0 to 100)
city_energy_profiles['suitability_score'] = (
    (city_energy_profiles['solar_norm'] * W_SOLAR) +
    (city_energy_profiles['wind_norm'] * W_WIND) +
    (city_energy_profiles['grid_norm'] * W_GRID)
) * 100

# Sort the final results to find the global winners!
final_rankings = city_energy_profiles.sort_values(by='suitability_score', ascending=False)

print("--- 🌍 TOP 5 ABSOLUTE BEST SITES FOR RENEWABLE DEPLOYMENT 🌍 ---")
print(final_rankings[['city', 'suitability_score', 'solar_annual_kwh_m2', 'wind_mean_ms', 'distance_to_grid_km']].head())

--- 🌍 TOP 5 ABSOLUTE BEST SITES FOR RENEWABLE DEPLOYMENT 🌍 ---
           city  suitability_score  solar_annual_kwh_m2  wind_mean_ms  \
45      Colombo          86.383600          1113.534571      4.387293   
43     Castries          74.940378           576.384857      7.036988   
146      Roseau          74.311365           571.109143      6.963664   
158  St Georges          73.891426           580.704286      7.017943   
131  Port Louis          73.556441           556.437714      6.811376   

     distance_to_grid_km  
45              3.123140  
43             54.595920  
146            55.695225  
158           145.214889  
131             1.111949  


In [15]:
import os

# Let's see exactly what files are inside your SRTM/Terrain folder
terrain_dir = "../datasets/srtm/"
print("--- Checking Terrain Folder Contents ---")
if os.path.exists(terrain_dir):
    files = os.listdir(terrain_dir)
    print("Files found in srtm folder:", files)
else:
    print("Terrain folder not found!")

--- Checking Terrain Folder Contents ---
Files found in srtm folder: ['dataset_v3.csv']


In [16]:
DEM_PATH = "../datasets/srtm/dataset_v3.csv"

print("--- Checking Terrain/DEM Dataset ---")
if os.path.exists(DEM_PATH):
    # Load just the first 5 rows to see the column names
    df_terrain = pd.read_csv(DEM_PATH, nrows=5)
    print("Success! Terrain file read correctly.")
    print("Columns available:", df_terrain.columns.tolist())
    print("\nFirst 3 rows preview:")
    print(df_terrain.head(3))
else:
    print(f"Error: File not found at {DEM_PATH}")

--- Checking Terrain/DEM Dataset ---
Success! Terrain file read correctly.
Columns available: ['Latitude', 'Longitude', 'LST_Day', 'LST_Night', 'LST_Diff', 'Emis_31', 'Emis_32', 'NDVI', 'EVI', 'NDWI', 'Sun_Angle', 'Albedo_Diff', 'View_Angle', 'Rel_Azimuth', 'DEM']

First 3 rows preview:
   Latitude  Longitude  LST_Day  LST_Night  LST_Diff  Emis_31  Emis_32  \
0      42.0  27.002363   311.74     295.38     16.36    0.984    0.986   
1      42.0  27.013577   312.30     295.30     17.00    0.984    0.988   
2      42.0  27.024791   312.68     295.60     17.08    0.984    0.986   

     NDVI     EVI      NDWI  Sun_Angle  Albedo_Diff  View_Angle  Rel_Azimuth  \
0  0.3197  0.2274  0.161246      33.45    -0.010020        2.86       150.15   
1  0.3233  0.2282  0.144767      33.45    -0.014092        2.88       150.12   
2  0.3941  0.2979  0.233109      36.53    -0.032945       27.66       -22.09   

     DEM  
0  348.0  
1  298.0  
2  312.0  


In [17]:
import pandas as pd
import numpy as np
import os

DEM_PATH = "../datasets/srtm/dataset_v3.csv"

# 1. Load the full terrain dataset
df_terrain_full = pd.read_csv(DEM_PATH)

# 2. Sort by coordinates to align the spatial grid properly
df_terrain_full = df_terrain_full.sort_values(by=['Latitude', 'Longitude']).reset_index(drop=True)

# 3. Calculate the elevation change (Rise) between neighboring points
df_terrain_full['next_DEM'] = df_terrain_full['DEM'].shift(-1)
df_terrain_full['next_Lat'] = df_terrain_full['Latitude'].shift(-1)
df_terrain_full['next_Lon'] = df_terrain_full['Longitude'].shift(-1)

# Drop the last row since it won't have a neighbor to compare against
df_terrain_full = df_terrain_full.dropna(subset=['next_DEM']).copy()

# 4. Haversine function to find horizontal distance (Run) in meters
def haversine_meters(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return c * 6371 * 1000 

run_meters = haversine_meters(
    df_terrain_full['Latitude'], df_terrain_full['Longitude'],
    df_terrain_full['next_Lat'], df_terrain_full['next_Lon']
)
run_meters = np.where(run_meters == 0, 0.001, run_meters) # Prevent division by zero

# 5. Calculate vertical rise
rise_meters = np.abs(df_terrain_full['DEM'] - df_terrain_full['next_DEM'])

# 6. Calculate slope angle in degrees: arctan(Rise / Run)
df_terrain_full['slope_degrees'] = np.degrees(np.arctan(rise_meters / run_meters))

# 7. Apply Engineering Safety Override Rule (Industry standard: >15° is too steep)
df_terrain_full['engineering_status'] = np.where(
    df_terrain_full['slope_degrees'] > 15.0, 
    'CRITICAL: Too Steep (Unsuitable)', 
    'NOMINAL: Safe Terrain'
)

print("--- ⛰️ Terrain Slope & Safety Verification ⛰️ ---")
print(df_terrain_full[['Latitude', 'Longitude', 'DEM', 'slope_degrees', 'engineering_status']].head(5))

print("\n--- Summary Breakdown of Land Safety Across the Tested Region ---")
print(df_terrain_full['engineering_status'].value_counts())

--- ⛰️ Terrain Slope & Safety Verification ⛰️ ---
    Latitude  Longitude    DEM  slope_degrees     engineering_status
0  40.008333  27.003470   97.0       4.012559  NOMINAL: Safe Terrain
1  40.008333  27.014350  162.0       0.123665  NOMINAL: Safe Terrain
2  40.008333  27.025229  164.0       2.163120  NOMINAL: Safe Terrain
3  40.008333  27.036109  199.0       0.618304  NOMINAL: Safe Terrain
4  40.008333  27.046989  189.0       2.410057  NOMINAL: Safe Terrain

--- Summary Breakdown of Land Safety Across the Tested Region ---
engineering_status
NOMINAL: Safe Terrain               23782
CRITICAL: Too Steep (Unsuitable)       32
Name: count, dtype: int64


In [18]:
# Create a folder for your processed output if it doesn't exist
import os
output_dir = "../datasets/processed"
os.makedirs(output_dir, exist_ok=True)

# Save the unified global city rankings
city_output_path = os.path.join(output_dir, "processed_global_city_rankings.csv")
city_energy_profiles.to_csv(city_output_path, index=False)

# Save the localized terrain safety analysis
terrain_output_path = os.path.join(output_dir, "processed_marmara_terrain_safety.csv")
df_terrain_full.to_csv(terrain_output_path, index=False)

print("--- 💾 Data Export Successful! 💾 ---")
print(f"1. Saved global rankings to: {city_output_path}")
print(f"2. Saved terrain safety layers to: {terrain_output_path}")

--- 💾 Data Export Successful! 💾 ---
1. Saved global rankings to: ../datasets/processed\processed_global_city_rankings.csv
2. Saved terrain safety layers to: ../datasets/processed\processed_marmara_terrain_safety.csv
